# Congress API

In [1]:
import json
import sys
import requests
import copy
import pandas as pd
from pathlib import Path

sys.path.append(str(Path('..').resolve()))
from my_secrets import secrets
from utility import congress_endpoint

Set up basic folder structure

In [2]:
# Base path (this notebook lives in its own Congress/ folder)
curr_dir = Path()

# Congress folders
congress_dir = curr_dir
congress_input_dir = congress_dir / 'input'
congress_output_dir = congress_dir / 'output'

## Bills

In [4]:
limit = 100

payload = {
  'format':'json',
  'offset':0,
  'limit':limit,
  'fromDateTime':'2026-01-01T00:00:00Z',
  'api_key':secrets['Congress']['headers']['Authorization']
}

bills = requests.get(
  congress_endpoint('bill'),
  params=payload
).json()['bills']

In [5]:
# Export input
congress_bills_sample_path = congress_input_dir / f'congress_bills_sample_{limit}.json'
with open(congress_bills_sample_path, mode='w') as f:
    f.write(json.dumps(bills, indent=4))

In [6]:
# Update 'bills' by reference
for bill in bills:
    bill['latestActionDate'] = bill['latestAction']['actionDate']
    bill['latestActionText'] = bill['latestAction']['text']
    del bill['latestAction']

In [7]:
bills = pd.DataFrame(bills)
bills.head()

,congress,number,originChamber,originChamberCode,title,type,updateDate,updateDateIncludingText,url,latestActionDate,latestActionText
0,108,5,House,H,"Entitled the ""English Plus Resolution"".",HCONRES,2026-03-23,2026-03-23,https://api.congress.gov/v3/bill/108/hconres/5...,2003-02-21,Referred to the Subcommittee on Education Reform.
1,108,32,House,H,Expressing the sense of the House of Represent...,HRES,2026-03-23,2026-03-23,https://api.congress.gov/v3/bill/108/hres/32?f...,2003-03-06,Referred to the Subcommittee on the Constitution.
2,108,85,House,H,Congratulating the Messiah College men's socce...,HRES,2026-03-23,2026-03-23,https://api.congress.gov/v3/bill/108/hres/85?f...,2003-03-10,Referred to the Subcommittee on 21st Century C...
3,108,86,House,H,Recognizing the contributions of historically ...,HRES,2026-03-23,2026-03-23,https://api.congress.gov/v3/bill/108/hres/86?f...,2003-03-10,Referred to the Subcommittee on 21st Century C...
4,108,116,House,H,Expressing the sense of the House of Represent...,HRES,2026-03-23,2026-03-23,https://api.congress.gov/v3/bill/108/hres/116?...,2003-03-17,Referred to the Subcommittee on Employer-Emplo...


In [8]:
# Export output
bills.to_csv(congress_output_dir / f'congress_bills_sample_{limit}.csv')

## Bill Details

In [9]:
# We have a preset number of records, so we don't need to use .append()
bill_details_list = [{}] * limit
bill_details_source_sample_list = [{}] * limit

for i, bill_detail_url in enumerate(bills.loc[:, 'url']):
    bill_details = requests.get(
        bill_detail_url,
        params={
            'api_key': secrets['Congress']['headers']['Authorization']
        }
    ).json()['bill']

    bill_details_source_sample_list[i] = copy.deepcopy(bill_details)

    # Already in the Bills table
    del bill_details['latestAction']


    # Unpack specific dictionaries into individual key-value pairs (i.e., columns later on)
    detail_keys = [
        'actions',
        'committees',
        'subjects',
        'summaries', # TODO: 'summaries' doesn't format nicely in the loop below
        'textVersions',
        'titles',
        'amendments',
        'relatedBills',
    ]

    for detail_key in detail_keys:
        new_count_key = f'{detail_key[:-1]}Count'
        new_url_key = f'{detail_key[:-1]}URL'
        try:
            # Count 
            bill_details[new_count_key] = bill_details[detail_key]['count']

            # URL
            bill_details[new_url_key] = bill_details[detail_key]['url']
            
            # Delete unpacked dict
            del bill_details[detail_key]
        except KeyError:
            bill_details[new_count_key] = None
            bill_details[new_url_key] = None


    # Policy area seems to just hold name
    # print(bill_details['policyArea']) # test it out
    bill_details['policyAreaName'] = bill_details['policyArea']['name']
    del bill_details['policyArea']

    # Only need the FK reference to the Member table
    bill_details['sponsorIds'] = [sponsor['bioguideId'] for sponsor in bill_details['sponsors']]
    del bill_details['sponsors']

    # Cosponsors
    try:
        bill_details['cosponsorCount'] = bill_details['cosponsors']['count']
        bill_details['countIncludingWithdrawnCosponsors'] = bill_details['cosponsors']['countIncludingWithdrawnCosponsors']
        bill_details['cosponsorURL'] = bill_details['cosponsors']['url']
        del bill_details['cosponsors']
    except KeyError: # Some bills have no cosponsors
        bill_details['cosponsorCount'] = None
        bill_details['countIncludingWithdrawnCosponsors'] = None
        bill_details['cosponsorURL'] = None

    # committeeReports
    try:
        bill_details['committeeReportCitations'] = [committeeReport['citation'] for committeeReport in bill_details['committeeReports']]
        bill_details['committeeReportURLs'] = [committeeReport['url'] for committeeReport in bill_details['committeeReports']]
        del bill_details['committeeReports']
    except KeyError: # Some bills have no committeeReports
        bill_details['committeeReportCitations'] = None
        bill_details['committeeReportURLs'] = None

    # notes
    try:
        bill_details['notesText'] = [committeeReport['text'] for committeeReport in bill_details['notes']]
        del bill_details['notes']
    except KeyError: # Some bills have no notes
        bill_details['notesText'] = None

    bill_details_list[i] = bill_details

In [10]:
with open(congress_input_dir / f'congress_bill_details_sample_{limit}.json', mode='w') as f:
    f.write(json.dumps(bill_details_source_sample_list, indent=4))

In [11]:
bill_details = pd.DataFrame(bill_details_list)
bill_details.head()

,congress,introducedDate,legislationUrl,number,originChamber,originChamberCode,title,type,updateDate,updateDateIncludingText,...,relatedBillCount,relatedBillURL,policyAreaName,sponsorIds,cosponsorCount,countIncludingWithdrawnCosponsors,cosponsorURL,committeeReportCitations,committeeReportURLs,notesText
0,108,2003-01-07,https://www.congress.gov/bill/108th-congress/h...,5,House,H,"Entitled the ""English Plus Resolution"".",HCONRES,2026-03-23T12:35:20Z,2026-03-23T12:35:20Z,...,NaN,NaN,Government Operations and Politics,[S000248],NaN,NaN,NaN,None,None,None
1,108,2003-01-27,https://www.congress.gov/bill/108th-congress/h...,32,House,H,Expressing the sense of the House of Represent...,HRES,2026-03-23T12:41:21Z,2026-03-23T12:41:21Z,...,NaN,NaN,"Civil Rights and Liberties, Minority Issues",[D000096],62.0,62.0,https://api.congress.gov/v3/bill/108/hres/32/c...,None,None,None
2,108,2003-02-13,https://www.congress.gov/bill/108th-congress/h...,85,House,H,Congratulating the Messiah College men's socce...,HRES,2026-03-23T12:44:53Z,2026-03-23T12:44:53Z,...,NaN,NaN,Commemorations,[P000585],11.0,11.0,https://api.congress.gov/v3/bill/108/hres/85/c...,None,None,None
3,108,2003-02-13,https://www.congress.gov/bill/108th-congress/h...,86,House,H,Recognizing the contributions of historically ...,HRES,2026-03-23T12:44:53Z,2026-03-23T12:44:53Z,...,NaN,NaN,Commemorations,[R000575],22.0,22.0,https://api.congress.gov/v3/bill/108/hres/86/c...,None,None,None
4,108,2003-02-27,https://www.congress.gov/bill/108th-congress/h...,116,House,H,Expressing the sense of the House of Represent...,HRES,2026-03-23T12:41:21Z,2026-03-23T12:41:21Z,...,NaN,NaN,Commemorations,[R000053],NaN,NaN,NaN,None,None,None


In [12]:
bill_details.to_csv(
    congress_output_dir / f'congress_bill_details_sample_{limit}.csv'
)

## Actions

In [13]:
bill_actions_list = [] # We don't know how many total actions for all bills there are
bill_actions_source_list = [[]] * limit # We'll want to save an example of what we used as a source

sourceSystemMapper = {
    'Senate': 0,
    'House committee actions': 1,
    'House floor actions': 2,
    'Library of Congress': 9
}

for i, bill_actions_url in enumerate(bill_details.loc[:, 'actionURL']):
    actions_request = requests.get(
        bill_actions_url,
        params={'api_key':secrets['Congress']['headers']['Authorization']}
    ).json()

    bill_info = actions_request['request'] # Acquire bill identifier
    bill_actions = actions_request['actions'] # Actions content

    bill_actions_source_list[i] = copy.deepcopy(bill_actions) # Not passing by reference
    
    for bill_action in bill_actions:
        try:
            # Just store the codes to join later
            bill_action['committeeCodes'] = [committee['systemCode'] for committee in bill_action['committees']]
            del bill_action['committees']
        except KeyError: # No committee when introduced from Library of Congress
            bill_action['committeeCodes'] = None

        try:
            # https://github.com/LibraryOfCongress/api.congress.gov/blob/main/Documentation/BillEndpoint.md#actions-level
            # sourceSystem can only have 4 values, which are outlined at link above
            bill_action['sourceSystemCode'] = bill_action['sourceSystem']['code']
            del bill_action['sourceSystem']
        except KeyError: # Some action sourceSystems don't contain a code key-value pair
            sourceSystemName = bill_action['sourceSystem']['name']
            bill_action['sourceSystemCode'] = sourceSystemMapper[sourceSystemName]

        bill_action['congress'] = bill_info['congress']
        bill_action['billNumber'] = bill_info['billNumber']
        bill_action['billType'] = bill_info['billType']

        bill_actions_list.append(bill_action)
    # print(json.dumps(bill_actions, indent=4))

In [14]:
with open(congress_input_dir / 'congress_bill_actions_sample.json', mode='w') as f:
    f.write(json.dumps(bill_actions_source_list, indent=4))

In [15]:
bill_actions = pd.DataFrame(bill_actions_list)
num_sample_bill_actions = len(bill_actions)

bill_actions.head()

,actionCode,actionDate,text,type,committeeCodes,sourceSystemCode,congress,billNumber,billType,sourceSystem,actionTime,recordedVotes,calendarNumber
0,H11000,2003-02-21,Referred to the Subcommittee on Education Reform.,Committee,[hsed14],1,108,5,hconres,NaN,NaN,NaN,NaN
1,H11100,2003-01-07,Referred to the House Committee on Education a...,IntroReferral,[hsed00],2,108,5,hconres,NaN,NaN,NaN,NaN
2,Intro-H,2003-01-07,Introduced in House,IntroReferral,None,9,108,5,hconres,NaN,NaN,NaN,NaN
3,1000,2003-01-07,Introduced in House,IntroReferral,None,9,108,5,hconres,NaN,NaN,NaN,NaN
4,H11000,2003-03-06,Referred to the Subcommittee on the Constitution.,Committee,[hsju10],1,108,32,hres,NaN,NaN,NaN,NaN


In [16]:
bill_actions.to_csv(congress_output_dir / 'congress_bill_actions_sample.csv')

## Amendments

In [33]:
bill_details.loc[bill_details.amendmentURL.notna(), ['amendmentCount', 'amendmentURL']].values[0]

array([1.0,
       'https://api.congress.gov/v3/bill/110/hconres/143/amendments?format=json'],
      dtype=object)

In [39]:
amendment_list = [] # We don't know how many amendments we'll receive per bill, nor how many bills have amendments at all
# TODO: Create a sample source file
    # ! difficult to do, since it's based on several endpoints...

for amendment_url in bill_details.loc[bill_details.amendmentURL.notna(), 'amendmentURL']:
    amendment = requests.get(
        amendment_url,
        params={'api_key':secrets['Congress']['headers']['Authorization']}
    ).json()

    bill_info = amendment['request']
    for amendment in amendment['amendments']:
        amendment['billNumber'] = bill_info['billNumber']
        amendment['billType'] = bill_info['billType']
        del amendment['latestAction'] # We already have all of the actions; And the latestAction can update over time

        # Get amended amendment if present
        try:
            amendment['amendedAmendmentNumber'] = amendment['amendedAmendment']['number']
            amendment['amendedAmendmentCongress'] = amendment['amendedAmendment']['congress']
            amendment['amendedAmendmentType'] = amendment['amendedAmendment']['type']
            del amendment['amendedAmendment']['congress']
        except KeyError:
            amendment['amendedAmendmentNumber'] = None
            amendment['amendedAmendmentCongress'] = None
            amendment['amendedAmendmentType'] = None

        # Amendment details
        amendment_details = requests.get(
            amendment['url'],
            params={'api_key':secrets['Congress']['headers']['Authorization']}
        ).json()['amendment']

        amendment['sponsorIds'] = [sponsor['bioguideId'] for sponsor in amendment_details['sponsors']]

        try: # Some amendments have cosponsors
            cosponsors = requests.get(
                amendment_details['cosponsors']['url'], # If not, this will throw a KeyError
                params={'api_key':secrets['Congress']['headers']['Authorization']}
            ).json()['cosponsors']
            amendment['cosponsorIds'] = [cosponsor['bioguidId'] for cosponsor in cosponsors]
            amendment['cosponsorCount'] = amendment_details['consponsors']['count']
            amendment['cosponsorCountIncludingWithdrawnCosponsors'] = amendment_details['consponsors']['countIncludingWithdrawnCosponsors']
        except KeyError:
            amendment['cosponsorIds'] = None
            amendment['cosponsorCount'] = None
            amendment['cosponsorCountIncludingWithdrawnCosponsors'] = None

        amendment['submittedDate'] = amendment_details['submittedDate']

        # Just get high level text information for now
        # TODO: Determine if the value of the 'textVersions' is a list when there are multiple versions
        amendment['textVersionCount'] = amendment_details['textVersions']['count']
        amendment['textVersionURL'] = amendment_details['textVersions']['url']

        # TODO: Find out what the difference between the two update date fields are
        amendment['amendmentDetailUpdateDate'] = amendment_details['updateDate']

        amendment_list.append(amendment)

In [40]:
amendments = pd.DataFrame(amendment_list)
amendments.head()

,congress,description,number,type,updateDate,url,billNumber,billType,amendedAmendmentNumber,amendedAmendmentCongress,amendedAmendmentType,sponsorIds,cosponsorIds,cosponsorCount,cosponsorCountIncludingWithdrawnCosponsors,submittedDate,textVersionCount,textVersionURL,amendmentDetailUpdateDate
0,110,Amendment in the nature of a substitute is tec...,721,HAMDT,2020-01-29T20:29:55Z,https://api.congress.gov/v3/amendment/110/hamd...,143,hconres,None,None,None,[D000096],None,None,None,2007-07-30T04:00:00Z,1,https://api.congress.gov/v3/amendment/110/hamd...,2022-02-03T05:17:20Z


In [42]:
# We won't know how many amendments we have, so I'm opting to not specify one
# ? Should we pass the length into the file name?
amendments.to_csv(congress_output_dir / 'congress_bill_amendments_sample.csv')

## Text

In [67]:
text_version_list = []
text_version_source_sample_list = []
for textVersionURL in bill_details.loc[:, 'textVersionURL']:
    textVersionContent = requests.get(
        textVersionURL,
        params={'api_key':secrets['Congress']['headers']['Authorization']}
    ).json()

    text_version_source_sample_list.append(copy.deepcopy(textVersionContent))

    bill_info = textVersionContent['request']

    # We will assign version numbers, where the latest is first in the list of text versions
    max_version = textVersionContent['pagination']['count']

    for i, textVersion in enumerate(textVersionContent['textVersions']):
        # Add version number
        textVersion['version'] = (max_version - i)

        # Keep only the links
        # TODO: Confirm that the order is consistent
        # TODO: Confirm that there are always 2
        # TODO: Confirm there are never 0 or more than 2
        textVersion['htm'] = textVersion['formats'][0] # assuming .htm is always first
        textVersion['pdf'] = textVersion['formats'][1] # assuming .htm is always second
        del textVersion['formats']
    
        # Bill identifiers
        textVersion['billNumber'] = bill_info['billNumber']
        textVersion['billType'] = bill_info['billType']
        textVersion['congress'] = bill_info['congress']

        text_version_list.append(textVersion)

In [71]:
with open(congress_input_dir / 'congress_text_versions_sample.json', mode='w') as f:
    f.write(json.dumps(text_version_source_sample_list, indent=4))

In [72]:
text_versions = pd.DataFrame(text_version_list)
text_versions.head()

,date,type,version,htm,pdf,billNumber,billType,congress
0,2003-01-07T05:00:00Z,Introduced in House,1,"{'type': 'Formatted Text', 'url': 'https://www...","{'type': 'PDF', 'url': 'https://www.congress.g...",5,hconres,108
1,2003-01-27T05:00:00Z,Introduced in House,1,"{'type': 'Formatted Text', 'url': 'https://www...","{'type': 'PDF', 'url': 'https://www.congress.g...",32,hres,108
2,2003-02-13T05:00:00Z,Introduced in House,1,"{'type': 'Formatted Text', 'url': 'https://www...","{'type': 'PDF', 'url': 'https://www.congress.g...",85,hres,108
3,2003-02-13T05:00:00Z,Introduced in House,1,"{'type': 'Formatted Text', 'url': 'https://www...","{'type': 'PDF', 'url': 'https://www.congress.g...",86,hres,108
4,2003-02-27T05:00:00Z,Introduced in House,1,"{'type': 'Formatted Text', 'url': 'https://www...","{'type': 'PDF', 'url': 'https://www.congress.g...",116,hres,108


In [73]:
text_versions.to_csv(congress_output_dir / 'congress_text_versions_sample.csv')